# Notebook 19 — development pilot report definitions

Purpose: turn committed profiling, training and development records into native Matplotlib figures and a portable HTML report for the three fixed conditions. This notebook defines reporting only; it reads no model, image or official-test pixels. Imports use only cells explicitly tagged `ahr-export`.

Inputs: `result`, profiler `rows`, training `arms`, and evaluator `initial_rows`, `matched_rows`, `rows`. Outputs: `report.html`, native PNGs, `result.json` and `descriptive_analysis.json` under the supplied artifact directory. Runtime: under one minute for a completed pilot. Checkpoints: this renderer has no model state and does not alter scientific checkpoints. Resume: rerun the caller with the same committed records and artifact path; atomic replacement refreshes the presentation without restarting training.

Two training seeds permit descriptive comparisons, not significance or superiority claims. UNKNOWN endpoints stay missing; paired ranges are observed seed spread and never confidence intervals. The final official dataset remains closed.

### Normalize certified development evidence

Retain every registered condition and endpoint. Linear scores require a complete, converged, numerically valid result. Optional kNN cutoffs do not invalidate the certified linear score.

In [ ]:
from pathlib import Path
import base64,html,json,math,os,time

CONDITIONS=('vit','learned_optimized','learned_global_prior')
LABELS={'vit':'ViT','learned_optimized':'Learned, initial local 50%','learned_global_prior':'Learned, initial local 10%','learned_original':'Original learned implementation'}
COLORS={'vit':'#315f8c','learned_optimized':'#c27228','learned_global_prior':'#6a6f32','learned_original':'#697880'}
ENDPOINTS=('initial','matched200','final')
ENDPOINT_LABELS={'initial':'Untrained','matched200':'200 updates','final':'900-second budget'}

def finite(value):
    return isinstance(value,(int,float)) and not isinstance(value,bool) and math.isfinite(value)

def number(value,digits=2):
    return ('%.'+str(digits)+'f')%value if finite(value) else 'UNKNOWN'

def records(value):
    if isinstance(value,dict):return list(value.values())
    return list(value or [])

def endpoint_name(value):
    return 'matched200' if value in ('matched','matched200','matched_updates','matched200updates') else value

def condition_name(row):
    for key in ('variant','condition','kind'):
        value=row.get(key)
        if value in CONDITIONS or value=='learned_original':return value
    arm_id=str(row.get('arm_id',''))
    for value in CONDITIONS:
        if arm_id.endswith('-'+value):return value
    return None

def seed_name(row):
    if row.get('seed') is not None:return int(row['seed'])
    arm_id=str(row.get('arm_id',''))
    if arm_id.startswith('s'):
        try:return int(arm_id.split('-')[0][1:])
        except ValueError:pass
    return None

def evaluation_rows(evaluation):
    """Use only explicit endpoint families; do not infer a milestone from final updates."""
    values=[]
    for key,endpoint in (('initial_rows','initial'),('matched_rows','matched200'),('rows','final'),('final_rows','final')):
        for source in records((evaluation or {}).get(key)):
            if not isinstance(source,dict):continue
            row=dict(source);row['report_endpoint']=endpoint_name(row.get('endpoint_label',endpoint))
            row['report_condition']=condition_name(row);row['report_seed']=seed_name(row)
            # Only certified and numerically valid primary rows are plotted.
            row['report_primary_valid']=(row.get('status')=='COMPLETE' and row.get('converged') is True and row.get('numerical_valid') is True and finite(row.get('accuracy_percent')))
            knn=row.get('knn_accuracy_percent',row.get('knn',{}).get('accuracy_percent'))
            row['report_knn_valid']=(row['report_primary_valid'] and finite(knn) and row.get('knn',{}).get('status','COMPLETE')=='COMPLETE' and not str(row.get('secondary_knn_status','')).startswith('UNKNOWN'))
            row['report_knn_accuracy']=knn if row['report_knn_valid'] else None
            values.append(row)
    unique={}
    for row in values:
        key=(row['report_seed'],row['report_condition'],row['report_endpoint'])
        if key in unique:
            # Ambiguous distinct endpoints are a visible integrity gap, never last-writer wins.
            if unique[key].get('model_id')!=row.get('model_id') or unique[key].get('accuracy_percent')!=row.get('accuracy_percent'):
                row=dict(row);row['status']='UNKNOWN_DUPLICATE_ENDPOINT';row['reason']='Distinct source rows share a report endpoint';row['report_primary_valid']=False;row['report_knn_valid']=False;row['report_knn_accuracy']=None
        unique[key]=row
    return list(unique.values())

def registered_seeds(result,evaluation=None,training=None):
    for source in (result,(result or {}).get('plan',{}),(result or {}).get('registration',{})):
        for key in ('seeds','paired_seeds','training_seeds'):
            if source.get(key):return [int(seed) for seed in source[key]]
    values={seed_name(row) for row in records((training or {}).get('arms'))}
    values.update(row['report_seed'] for row in evaluation_rows(evaluation))
    return sorted(seed for seed in values if seed is not None)

def analyze(result,evaluation=None,training=None):
    """Descriptive development evidence; exactly zero hypothesis tests or confidence intervals."""
    rows=evaluation_rows(evaluation);seeds=registered_seeds(result,evaluation,training)
    lookup={(row['report_seed'],row['report_condition'],row['report_endpoint']):row for row in rows}
    expected=[]
    for seed in seeds:
        for condition in CONDITIONS:
            for endpoint in ENDPOINTS:
                source=lookup.get((seed,condition,endpoint),{})
                expected.append({'seed':seed,'condition':condition,'endpoint':endpoint,'status':source.get('status','UNKNOWN_NOT_YET_OBSERVED'),
                  'reason':source.get('reason','Endpoint not observed') if not source else source.get('reason',''),
                  'accuracy_percent':source.get('accuracy_percent') if source.get('report_primary_valid') else None,
                  'knn_accuracy_percent':source.get('report_knn_accuracy') if source.get('report_knn_valid') else None,
                  'knn_status':source.get('secondary_knn_status',source.get('knn',{}).get('status','UNKNOWN')),
                  'updates':source.get('updates',source.get('actual_updates')),
                  'population_size':source.get('population_size',source.get('query_count',source.get('num_examples'))),
                  'model_id':source.get('model_id'),'primary_readout_status':source.get('primary_readout_status',source.get('status','UNKNOWN'))})
    effects=[];gains=[]
    for endpoint in ('matched200','final'):
        for condition in CONDITIONS[1:]:
            for metric in ('accuracy_percent','knn_accuracy_percent'):
                values=[]
                for seed in seeds:
                    arm=next(row for row in expected if (row['seed'],row['condition'],row['endpoint'])==(seed,condition,endpoint))
                    ref=next(row for row in expected if (row['seed'],row['condition'],row['endpoint'])==(seed,'vit',endpoint))
                    delta=arm[metric]-ref[metric] if finite(arm[metric]) and finite(ref[metric]) else None
                    effects.append({'seed':seed,'condition':condition,'endpoint':endpoint,'metric':metric,'delta_pp':delta,'status':'MEASURED' if finite(delta) else 'UNKNOWN'})
        for condition in CONDITIONS:
            for seed in seeds:
                initial=next(row for row in expected if (row['seed'],row['condition'],row['endpoint'])==(seed,condition,'initial'))
                arm=next(row for row in expected if (row['seed'],row['condition'],row['endpoint'])==(seed,condition,endpoint))
                gains.append({'seed':seed,'condition':condition,'endpoint':endpoint,**{metric+'_gain_pp':arm[metric]-initial[metric] if finite(arm[metric]) and finite(initial[metric]) else None for metric in ('accuracy_percent','knn_accuracy_percent')}})
    return {'schema':'fmm19-descriptive-report/v1','seeds':seeds,'expected_rows':expected,'paired_effects':effects,'initial_control_gains':gains,
      'inferential_status':'EXPLORATORY_ONLY_TWO_PAIRED_SEEDS','confidence_intervals':None,'p_values':None,'scientific_superiority_established':False,'official_test_pixels_read':0}

def atomic_text(path,text):
    path=Path(path);temporary=path.with_name(path.name+'.tmp.'+str(os.getpid()))
    with temporary.open('w') as handle:handle.write(text);handle.flush();os.fsync(handle.fileno())
    os.replace(temporary,path)

def html_table(headers,rows):
    return '<div class="table-scroll"><table><thead><tr>'+''.join('<th>'+html.escape(str(header))+'</th>'for header in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+html.escape(str(value))+'</td>'for value in row)+'</tr>'for row in rows)+'</tbody></table></div>'


### Render native plots and a portable report

Whole JEPA costs, paired development endpoints and learning versus actual work use explicit source fields. Profiling self CUDA times describe accumulated operator work; they do not masquerade as an additive wall-time breakdown.

In [ ]:
def render(root,artifact,result,profiling=None,evaluation=None,training=None):
    """Regenerate source-bound native figures and standalone HTML without loading models/images."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    analysis=analyze(result,evaluation,training);seeds=analysis['seeds'];expected=analysis['expected_rows'];figures=[]
    fixture=bool(result.get('fixture_source'));prefix='Software fixture — ' if fixture else ''
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':False})
    def save(fig,name):
        fig.savefig(artifact/name,dpi=145,bbox_inches='tight');plt.close(fig);figures.append(name)
    def tidy(ax,ylabel,zero=True):
        ax.set_ylabel(ylabel);ax.grid(axis='y',alpha=.18)
        if zero:
            maximum=float(ax.dataLim.ymax)
            ax.set_ylim(0,max(1e-12,maximum*1.08)if math.isfinite(maximum)and maximum>0 else 1.)
        else:
            low,high=ax.get_ylim();ax.set_ylim(low,high+max(1e-12,high-low)*.14)
    def point_range(ax,x,values,color,label=None):
        values=[float(value) for value in values if finite(value)]
        if not values:return
        offsets=np.linspace(-.08,.08,len(values)) if len(values)>1 else [0]
        ax.scatter(np.asarray(offsets)+x,values,s=45,facecolors='none',edgecolors=color,label=label)
        ax.plot([x,x],[min(values),max(values)],color=color,linewidth=1.4)
        ax.scatter([x],[float(np.mean(values))],color=color,marker='D',s=38)
    # Profile rows are independent complete JEPA measurements, not attention-only microbenchmarks.
    profile_rows=[row for row in records((profiling or {}).get('rows')) if isinstance(row,dict) and row.get('status') in ('COMPLETE','MEASURED','PASS')]
    if profile_rows:
        measured=[]
        for row in profile_rows:
            condition=condition_name(row)
            if condition is None:
                condition=row.get('variant',row.get('condition',row.get('name','UNKNOWN')))
            condition={'original':'learned_original','original18':'learned_original','learned_baseline':'learned_original','learned_kernel':'learned_original','optimized19':'learned_optimized','optimized19_gate01':'learned_global_prior'}.get(condition,condition)
            timing=row.get('step_seconds',row.get('median_step_seconds',row.get('median_seconds_per_optimizer_update',row.get('total_step_seconds'))))
            if timing is None and finite(row.get('step_ms')):timing=row['step_ms']/1000
            peak=row.get('peak_memory_bytes',row.get('peak_allocated_bytes',row.get('max_memory_allocated_bytes')))
            measured.append({**row,'report_condition':condition,'report_step_seconds':timing,'report_peak_gib':peak/2**30 if finite(peak) else None})
        labels=[LABELS.get(row['report_condition'],str(row['report_condition'])) for row in measured]
        fig,axes=plt.subplots(1,2,figsize=(12,4.6),layout='constrained')
        for index,row in enumerate(measured):
            color=COLORS.get(row['report_condition'],'#697880')
            for ax,key in zip(axes,('report_step_seconds','report_peak_gib')):
                if finite(row[key]):ax.bar(index,row[key],color=color,width=.6);ax.annotate(number(row[key],3),(index,row[key]),xytext=(0,5),textcoords='offset points',ha='center',fontsize=9)
        for ax,title,ylabel in zip(axes,('Whole JEPA optimizer step','Whole-step peak CUDA allocation'),('Seconds per optimizer update','Peak allocated memory (GiB)')):
            ax.set_title(title);ax.set_xticks(range(len(labels)),labels,rotation=18,ha='right');tidy(ax,ylabel)
        fig.suptitle(prefix+'Complete JEPA profiling: fixed synthetic inputs, FP32; excludes data loading and checkpoint IO')
        save(fig,'profiling_whole_step.png')
        # Top operators are displayed per run: accumulated profiler time is not an additive wall breakdown.
        operator_groups=[]
        for row in measured:
            operations=row.get('operators',row.get('operator_rows',[]))
            operations=records(operations)
            observed=[]
            for op in operations:
                if not isinstance(op,dict):continue
                duration=op.get('self_cuda_time_ms',op.get('self_device_time_ms',op.get('self_device_ms',op.get('self_cuda_time_total_ms'))))
                if duration is None and finite(op.get('self_cuda_time_total')):duration=op['self_cuda_time_total']/1000
                if finite(duration):observed.append((str(op.get('name',op.get('operator',op.get('key','Unknown operator')))),duration))
            if observed:operator_groups.append((row['report_condition'],sorted(observed,key=lambda value:value[1],reverse=True)[:6]))
        if operator_groups:
            fig,axes=plt.subplots(len(operator_groups),1,figsize=(12,2.6*len(operator_groups)),squeeze=False,layout='constrained')
            for ax,(condition,operations) in zip(axes[:,0],operator_groups):
                labels=[value[0] for value in operations];values=[value[1]for value in operations]
                ax.barh(range(len(values)),values,color=COLORS.get(condition,'#697880'));ax.invert_yaxis();ax.set_yticks(range(len(labels)),labels);ax.set_xlabel('Accumulated self CUDA time (ms)');ax.set_title(LABELS.get(condition,condition));ax.set_xlim(left=0);ax.grid(axis='x',alpha=.18)
            fig.suptitle(prefix+'Profiler operator costs; accumulated kernel time is not wall time')
            save(fig,'profiling_operators.png')
    valid=any(finite(row['accuracy_percent']) or finite(row['knn_accuracy_percent']) for row in expected)
    if valid:
        fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
        for condition in CONDITIONS:
            for seed_index,seed in enumerate(seeds):
                group=[next(row for row in expected if(row['seed'],row['condition'],row['endpoint'])==(seed,condition,endpoint)) for endpoint in ENDPOINTS]
                for ax,key in zip(axes,('accuracy_percent','knn_accuracy_percent')):
                    values=[row[key] if finite(row[key]) else np.nan for row in group]
                    if np.any(np.isfinite(values)):
                        ax.plot(range(3),values,marker='o' if seed_index==0 else 's',linestyle='-' if seed_index==0 else '--',color=COLORS[condition],label=LABELS[condition]+', seed '+str(seed_index+1))
        for ax,title in zip(axes,('Certified linear readout','Cosine nearest-neighbour readout')):
            ax.set_title(title);ax.set_xticks(range(3),[ENDPOINT_LABELS[e]for e in ENDPOINTS],rotation=10);tidy(ax,'DEV top-1 accuracy (%)')
            if ax.lines:ax.legend(fontsize=8)
        fig.suptitle(prefix+'Development learning endpoints; missing or uncertified scores remain gaps')
        save(fig,'development_accuracy.png')
        fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
        categories=[(endpoint,condition)for endpoint in ('matched200','final')for condition in CONDITIONS[1:]]
        for ax,metric,title in zip(axes,('accuracy_percent','knn_accuracy_percent'),('Linear accuracy paired difference','Nearest-neighbour paired difference')):
            for index,(endpoint,condition)in enumerate(categories):
                values=[row['delta_pp']for row in analysis['paired_effects']if(row['endpoint'],row['condition'],row['metric'])==(endpoint,condition,metric)]
                point_range(ax,index,values,COLORS[condition])
                count=sum(finite(value)for value in values);ax.text(index,.98,'n='+str(count)+'/'+str(len(seeds)),transform=ax.get_xaxis_transform(),ha='center',va='top',fontsize=9,color='#586068')
            ax.axhline(0,color='#586068',linewidth=1);ax.set_title(title);ax.set_xticks(range(4),['200 updates\nLocal 50%','200 updates\nLocal 10%','900 seconds\nLocal 50%','900 seconds\nLocal 10%']);tidy(ax,'Candidate − paired ViT (percentage points)',False)
        fig.suptitle(prefix+'Exploratory paired effects: dots = seeds, diamond = mean, line = observed range; no confidence intervals')
        save(fig,'paired_effects.png')
        fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
        categories=[(endpoint,condition) for endpoint in ('matched200','final')for condition in CONDITIONS]
        for ax,metric,title in zip(axes,('accuracy_percent','knn_accuracy_percent'),('Linear improvement over own initialization','Nearest-neighbour improvement over own initialization')):
            for index,(endpoint,condition)in enumerate(categories):
                values=[row[metric+'_gain_pp']for row in analysis['initial_control_gains']if(row['endpoint'],row['condition'])==(endpoint,condition)]
                point_range(ax,index,values,COLORS[condition])
                count=sum(finite(value)for value in values);ax.text(index,.98,'n='+str(count)+'/'+str(len(seeds)),transform=ax.get_xaxis_transform(),ha='center',va='top',fontsize=9,color='#586068')
            ax.axhline(0,color='#586068',linewidth=1);ax.set_title(title);ax.set_xticks(range(6),['200u\nViT','200u\n50%','200u\n10%','900s\nViT','900s\n50%','900s\n10%']);tidy(ax,'Trained − own untrained control (percentage points)',False)
        fig.suptitle(prefix+'Initial controls isolate learning gains; ranges are observed seed spread, not inferential uncertainty')
        save(fig,'initial_control_gains.png')
    arms=records((training or {}).get('arms'))
    if arms:
        # One point per arm; missing compute/data/memory metrics stay absent and labelled unavailable.
        metrics=[('actual_updates','Successful optimizer updates'),('active_wall_seconds','Active training wall time (s)'),('complete_jepa_step_seconds','Median complete JEPA update (s)'),('jepa_compute_seconds','Median JEPA compute per update (s)'),('data_preprocessing_seconds','Median data preprocessing per update (s)'),('peak_cuda_allocated_bytes','Peak allocated memory (GiB)')]
        fig,axes=plt.subplots(2,3,figsize=(13,8),layout='constrained');axes=axes.ravel()
        for ax,(key,ylabel)in zip(axes,metrics):
            count=0
            for index,condition in enumerate(CONDITIONS):
                group=[row for row in arms if condition_name(row)==condition]
                values=[row.get(key,row.get('performance',{}).get(key))for row in group]
                values=[value.get('median')if isinstance(value,dict)else value for value in values]
                if key=='peak_cuda_allocated_bytes':values=[value/2**30 if finite(value) else None for value in values]
                count+=sum(finite(value)for value in values);point_range(ax,index,values,COLORS[condition])
            ax.set_xticks(range(3),['ViT','Local 50%','Local 10%']);tidy(ax,ylabel)
            if not count:ax.text(.5,.5,'UNKNOWN: metric not recorded',transform=ax.transAxes,ha='center',va='center',color='#697880')
        fig.suptitle(prefix+'Actual training work: observed arm values and seed ranges')
        save(fig,'training_work.png')
        if valid:
            fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
            for arm in arms:
                seed=seed_name(arm);condition=condition_name(arm)
                if seed not in seeds or condition not in CONDITIONS:continue
                matched=arm.get('matched_checkpoint',{})
                for ax,key in zip(axes,('actual_updates','active_wall_seconds')):
                    xs=[0,matched.get('updates') if key=='actual_updates' else matched.get('active_wall_seconds'),arm.get(key)]
                    ys=[next(row for row in expected if(row['seed'],row['condition'],row['endpoint'])==(seed,condition,endpoint))['accuracy_percent'] for endpoint in ENDPOINTS]
                    xx=[value if finite(value) else np.nan for value in xs];yy=[value if finite(value) else np.nan for value in ys]
                    if np.any(np.isfinite(xx)&np.isfinite(yy)):
                        seed_index=seeds.index(seed);ax.plot(xx,yy,marker='o'if seed_index==0 else's',linestyle='-'if seed_index==0 else'--',color=COLORS[condition],label=LABELS[condition]+', seed '+str(seed_index+1))
            for ax,xlabel in zip(axes,('Successful optimizer updates','Observed active training wall seconds')):
                ax.set_xlabel(xlabel);ax.set_xlim(left=0);tidy(ax,'Certified linear DEV accuracy (%)')
                if ax.lines:ax.legend(fontsize=8)
            fig.suptitle(prefix+'Learning versus actual work; milestone wall time is shown only when recorded')
            save(fig,'learning_by_work.png')
    diagnostics=[]
    for arm in arms:
        for row in records(arm.get('learned_diagnostics',arm.get('gate_temperature_diagnostics',[]))):
            if isinstance(row,dict):diagnostics.append({**row,'report_condition':condition_name(arm),'report_seed':seed_name(arm)})
    if diagnostics:
        metrics=[('gate_mean','Mean local mixing weight'),('temperature_mean','Mean feature temperature'),('gate_gradient_norm','Gate gradient norm'),('temperature_gradient_norm','Temperature gradient norm')]
        fig,axes=plt.subplots(2,2,figsize=(12,7),layout='constrained')
        for ax,(key,ylabel)in zip(axes.ravel(),metrics):
            count=0
            for condition in CONDITIONS[1:]:
                for seed in seeds:
                    group=sorted([row for row in diagnostics if row['report_condition']==condition and row['report_seed']==seed and finite(row.get('updates',row.get('step'))) and finite(row.get(key))],key=lambda row:row.get('updates',row.get('step')))
                    if group:
                        ax.plot([row.get('updates',row.get('step'))for row in group],[row[key]for row in group],marker='o'if seeds.index(seed)==0 else's',linestyle='-'if seeds.index(seed)==0 else'--',color=COLORS[condition],label=LABELS[condition]+', seed '+str(seeds.index(seed)+1));count+=len(group)
            ax.set_xlabel('Successful optimizer updates');tidy(ax,ylabel)
            if not count:ax.text(.5,.5,'UNKNOWN: metric not recorded',transform=ax.transAxes,ha='center',va='center')
            else:ax.legend(fontsize=8)
        fig.suptitle(prefix+'Recorded learned mixing and temperature diagnostics; missing records stay unavailable')
        save(fig,'learned_diagnostics.png')
    body=['<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Notebook 19: compute and local-gate development pilot</title>',
      '<style>body{font:16px system-ui;max-width:1180px;margin:32px auto;padding:0 24px;color:#172330}h1{font-size:28px}img{width:100%;height:auto}figure{margin:24px 0}figcaption,p{line-height:1.55}pre{white-space:pre-wrap;word-break:break-word;font:12px monospace}.table-scroll{overflow:auto}table{border-collapse:collapse;min-width:100%;font-size:13px}th,td{text-align:left;border-bottom:1px solid #d9dfe4;padding:7px}th{background:#f1f4f6}.notice{border-left:4px solid #c27228;padding:8px 14px;background:#faf5eb}</style></head><body>',
      '<h1>JEPA compute and local-gate development pilot</h1>',
      '<p><strong>Status: '+html.escape(str(result.get('status','UNKNOWN')))+'.</strong> '+html.escape(str(result.get('reason',result.get('phase',result.get('decision','')))))+'</p>',
      '<p>Three fixed conditions: ViT, optimized learned kernel with initial local weight 50%, and the same optimized kernel with initial local weight 10%. Two paired training seeds; FP32; image size 256; rank 64. Each arm has a 900-second active training budget and a separately retained 200-update endpoint when reached. Both candidates add 98,328 online parameters.</p>',
      '<p class="notice"><strong>Exploratory development evidence.</strong> This pilot cannot establish statistical superiority. Diamonds are means; range lines show only the observed training-seed spread, with no confidence intervals or significance tests. TRAIN-only certified linear classifiers and cosine nearest-neighbour readouts evaluate the same DEV population. Official test pixels remain closed.</p>']
    if fixture:body.append('<p class="notice"><strong>SOFTWARE VALIDATION ONLY. All displayed values are synthetic presentation fixtures, not scientific measurements.</strong></p>')
    if result.get('deadline_unix'):
        from datetime import datetime
        from zoneinfo import ZoneInfo
        deadline=datetime.fromtimestamp(result['deadline_unix'],ZoneInfo('Europe/Dublin')).strftime('%d %B %Y %H:%M %Z')
        body.append('<p>Original deadline: '+html.escape(deadline)+'. UNKNOWN cutoffs retain missing values; the timer is never reset by this report.</p>')
    if (profiling or {}).get('rows'):
        body.append('<h2>Complete JEPA profiling status</h2><p>Discarded weights and fixed synthetic masks. Unprofiled repeats estimate step timing; profiler self times describe accumulated operator work. Real data preparation and checkpoint costs belong to the pilot wall timers.</p>')
        body.append(html_table(['Implementation','Status','Median update (s)','Peak allocated (GiB)','Scope'],[[LABELS.get({'original18':'learned_original','optimized19':'learned_optimized','optimized19_gate01':'learned_global_prior'}.get(row.get('variant'),row.get('variant')),str(row.get('variant','UNKNOWN'))),row.get('status','UNKNOWN'),number(row.get('median_seconds_per_optimizer_update',row.get('step_seconds')),3),number(row['peak_allocated_bytes']/2**30)if finite(row.get('peak_allocated_bytes'))else'UNKNOWN',row.get('measurement_scope',row.get('reason','UNKNOWN'))]for row in records(profiling.get('rows')) if isinstance(row,dict)]))
    for name in figures:
        body.append('<figure><img alt="'+html.escape(name.replace('_',' ').replace('.png',''))+'" src="data:image/png;base64,'+base64.b64encode((artifact/name).read_bytes()).decode()+'"></figure>')
    body.append('<h2>Every registered development endpoint</h2><p>UNKNOWN includes missing, timed out, numerically invalid or uncertified readouts. A complete linear score may coexist with UNKNOWN nearest-neighbour scoring.</p>')
    if expected:
        body.append(html_table(['Seed','Condition','Endpoint','Status','Linear (%)','kNN (%)','kNN status'],[[row['seed'],LABELS[row['condition']],ENDPOINT_LABELS[row['endpoint']],row['status'],number(row['accuracy_percent']),number(row['knn_accuracy_percent']),row['knn_status']]for row in expected]))
    else:body.append('<p>No registered endpoint observations are available yet.</p>')
    body.append('<h2>Paired differences and initial controls</h2>')
    if analysis['paired_effects']:
        body.append(html_table(['Seed','Candidate','Endpoint','Readout','Candidate − ViT (pp)','Status'],[[row['seed'],LABELS[row['condition']],ENDPOINT_LABELS[row['endpoint']],'Linear'if row['metric']=='accuracy_percent'else'kNN',number(row['delta_pp']),row['status']]for row in analysis['paired_effects']]))
    body.append('<details><summary>Source records, parameter differences and integrity limits</summary><pre>'+html.escape(json.dumps({'result':result,'profiling':profiling,'evaluation':evaluation,'training':training,'descriptive_analysis':analysis},indent=2,allow_nan=False,default=str))+'</pre></details></body></html>')
    atomic_text(artifact/'report.html','\n'.join(body));atomic_text(artifact/'result.json',json.dumps(result,indent=2,allow_nan=False,default=str));atomic_text(artifact/'descriptive_analysis.json',json.dumps(analysis,indent=2,allow_nan=False,default=str))
    return {'report':str(artifact/'report.html'),'figures':figures,'status':result.get('status','UNKNOWN'),'analysis':str(artifact/'descriptive_analysis.json'),'inferential_status':analysis['inferential_status'],'official_test_pixels_read':0}
